# Experiment 2 - Data 2021


## Install & Import Library

In [ ]:
# install libraryt
!pip install statsmodels yfinance pandas-datareader requests scikit-learn xgboost lightgbm optuna plotly --quiet

In [ ]:
# import some libarries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
import os
import json
import pickle
import random

import requests
import re
import yfinance as yf
from datetime import datetime

from openpyxl import load_workbook

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

import xgboost as xgb
import lightgbm as lgb
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

In [ ]:
# set reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed = 42
set_seed(seed)

## Konfigurasi

In [ ]:
# path dataset
PATH_ANTAM = '/content/harga_emas_antm.csv'
PATH_XAU = '/content/Data Historis XAU_USD.csv'
PATH_KURS = '/content/data kurs usd idr.xlsx'
PATH_BI = '/content/BI-7Day-RR.xlsx'
PATH_INFLASI = '/content/Data Inflasi.xlsx'

In [ ]:
# config crawling, feature engineering params modeling
START_DATE = '2019-01-01'
END_DATE = '2026-02-20'
TRAIN_RATIO = 0.70
MAX_HORIZON = 7
SARIMAX_ORDER = (1, 1, 1)
SARIMAX_SEAS = (0, 0, 0, 0)
N_OPTUNA_TRIALS = 50

## Load & Merge Dataset

In [ ]:
BULAN_ID = {
    'Januari':'01','Februari':'02','Maret':'03','April':'04',
    'Mei':'05','Juni':'06','Juli':'07','Agustus':'08',
    'September':'09','Oktober':'10','November':'11','Desember':'12'
}

def parse_tgl_id(s):
    p = s.strip().split()
    return pd.to_datetime(f"{p[2]}-{BULAN_ID[p[1]]}-{p[0].zfill(2)}")

def parse_periode_id(s):
    p = s.strip().split()
    return pd.to_datetime(f"{p[1]}-{BULAN_ID[p[0]]}-01")

# 1. Harga Emas Antam
df_antam = pd.read_csv(PATH_ANTAM)
df_antam['tanggal'] = pd.to_datetime(df_antam['tanggal'])
df_antam = df_antam[['tanggal','harga']].rename(columns={'harga':'harga_emas_antam_idr'}).set_index('tanggal').sort_index()

# 2. XAU/USD (Investing.com)
df_xau = pd.read_csv(PATH_XAU)
df_xau['Tanggal'] = pd.to_datetime(df_xau['Tanggal'], format='%d/%m/%Y')
df_xau['harga_emas_dunia_usd'] = (df_xau['Terakhir']
    .str.replace('.','',regex=False).str.replace(',','.',regex=False).astype(float))
df_xau = df_xau[['Tanggal','harga_emas_dunia_usd']].set_index('Tanggal').sort_index()

# 3. Kurs USD/IDR (yfinance)
df_kurs = pd.read_excel(PATH_KURS)
df_kurs.drop('Unnamed: 0', axis=1, inplace=True)
df_kurs.columns = ['tanggal','kurs_usdidr']
df_kurs['tanggal'] = pd.to_datetime(df_kurs['tanggal'])
df_kurs = df_kurs[['tanggal','kurs_usdidr']].set_index('tanggal').sort_index()

# 4. BI 7-Day RR Rate (Bank Indonesia)
wb_bi = load_workbook(PATH_BI, read_only=True); ws_bi = wb_bi.active
bi_rows = []
for row in ws_bi.iter_rows(values_only=True):
    if row[0] is not None and isinstance(row[0], int):
        bi_rows.append({'tanggal': parse_tgl_id(row[1]),
                        'bi_7drr_rate': float(row[2].replace('%','').strip())})
df_bi = pd.DataFrame(bi_rows).set_index('tanggal').sort_index()

# 5. Inflasi YoY (Bank Indonesia)
wb_inf = load_workbook(PATH_INFLASI, read_only=True); ws_inf = wb_inf.active
inf_rows = []
for row in ws_inf.iter_rows(values_only=True):
    if row[0] is not None and isinstance(row[0], int):
        inf_rows.append({'tanggal': parse_periode_id(row[1]),
                         'inflasi_yoy_id': float(row[2].replace('%','').strip())})
df_inflasi = pd.DataFrame(inf_rows).set_index('tanggal').sort_index()

In [ ]:
# fitur yang digunakan untuk model boosting
FEATURE_COLS = [
    'log_return',
    'lag_1', 'lag_2', 'lag_3', 'lag_5', 'lag_7',
    'rolling_mean_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_std_14',
    'log_xau', 'log_kurs', 'log_xau_lag1', 'log_kurs_lag1',
    'bi_7drr_rate', 'inflasi_yoy_id', 'spread_macro',
    'day_of_week', 'month', 'quarter'
]

# eksogen SARIMAX: XAU/USD & Kurs saja
EXOG_SARIMAX = ['log_xau', 'log_kurs']

print(f'SARIMAX exogenous : {EXOG_SARIMAX}')
print(f'Boosting features : {len(FEATURE_COLS)} kolom')


SARIMAX exogenous : ['log_xau', 'log_kurs']
Boosting features : 20 kolom


In [ ]:
# helper function
def evaluate_forecast(y_true, y_pred):
    return {
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'r2': r2_score(y_true, y_pred)
    }

tscv = TimeSeriesSplit(n_splits=5)


In [ ]:
# merge data
START_DATE_EXP2 = '2021-01-01'
END_DATE_EXP2 = END_DATE  # '2026-02-20'

full_idx_e2 = pd.date_range(start=START_DATE_EXP2, end=END_DATE_EXP2, freq='D')

df_e2 = pd.DataFrame(index=full_idx_e2)
df_e2.index.name = 'tanggal'

df_e2 = df_e2.join(df_antam[df_antam.index >= START_DATE_EXP2])
df_e2 = df_e2.join(df_xau[df_xau.index >= START_DATE_EXP2])
df_e2 = df_e2.join(df_kurs[df_kurs.index >= START_DATE_EXP2])
df_e2 = df_e2.join(df_bi.reindex(full_idx_e2).ffill().bfill())
df_e2 = df_e2.join(df_inflasi.reindex(full_idx_e2).ffill().bfill())
df_e2 = df_e2.ffill().bfill()

print(f'Shape : {df_e2.shape}')
print(f'Periode: {df_e2.index[0].date()} s/d {df_e2.index[-1].date()}')
print(f'Missing:\n{df_e2.isnull().sum()}')

Shape : (1877, 5)
Periode: 2021-01-01 s/d 2026-02-20
Missing:
harga_emas_antam_idr    0
harga_emas_dunia_usd    0
kurs_usdidr             0
bi_7drr_rate            0
inflasi_yoy_id          0
dtype: int64


## Uji Stasioneritas (ADF Test)

In [ ]:
# function adf test (cek stasioner)
def adf_test(series, name):
    res = adfuller(series.dropna(), autolag='AIC')
    status = 'Stasioner' if res[1] < 0.05 else 'Tidak Stasioner'
    print(f'{name:<35} | ADF: {res[0]:>8.4f} | p: {res[1]:.4f} | {status}')

print('Level:')
adf_test(df_e2['harga_emas_antam_idr'], 'Harga Emas Antam (level)')
adf_test(df_e2['harga_emas_dunia_usd'], 'Harga Emas Dunia (level)')
adf_test(df_e2['kurs_usdidr'], 'Kurs USD/IDR (level)')

df_e2['log_harga'] = np.log(df_e2['harga_emas_antam_idr'])
df_e2['log_xau'] = np.log(df_e2['harga_emas_dunia_usd'])
df_e2['log_kurs'] = np.log(df_e2['kurs_usdidr'])

print('\nLog')
adf_test(df_e2['log_harga'], 'log(Harga Emas Antam)')
adf_test(df_e2['log_xau'], 'log(Harga Emas Dunia)')
adf_test(df_e2['log_kurs'], 'log(Kurs USD/IDR)')

print('\nLog First Difference')
adf_test(df_e2['log_harga'].diff().dropna(), 'log(Harga Emas Antam)')
adf_test(df_e2['log_xau'].diff().dropna(), 'log(Harga Emas Dunia)')
adf_test(df_e2['log_kurs'].diff().dropna(), 'log(Kurs USD/IDR)')

Level:
Harga Emas Antam (level)            | ADF:   5.5956 | p: 1.0000 | Tidak Stasioner
Harga Emas Dunia (level)            | ADF:   5.4082 | p: 1.0000 | Tidak Stasioner
Kurs USD/IDR (level)                | ADF:  -1.0678 | p: 0.7277 | Tidak Stasioner

Log
log(Harga Emas Antam)               | ADF:   3.5907 | p: 1.0000 | Tidak Stasioner
log(Harga Emas Dunia)               | ADF:   2.8808 | p: 1.0000 | Tidak Stasioner
log(Kurs USD/IDR)                   | ADF:  -1.1516 | p: 0.6941 | Tidak Stasioner

Log First Difference
log(Harga Emas Antam)               | ADF: -17.2667 | p: 0.0000 | Stasioner
log(Harga Emas Dunia)               | ADF: -17.3935 | p: 0.0000 | Stasioner
log(Kurs USD/IDR)                   | ADF: -27.7690 | p: 0.0000 | Stasioner


## Feature Engineering

In [ ]:
# feature engineering ttp
df_e2['log_harga'] = np.log(df_e2['harga_emas_antam_idr'])
df_e2['log_xau'] = np.log(df_e2['harga_emas_dunia_usd'])
df_e2['log_kurs'] = np.log(df_e2['kurs_usdidr'])

# Log return
df_e2['log_return'] = df_e2['log_harga'].diff()

# Lag features
for lag in [1, 2, 3, 5, 7]:
    df_e2[f'lag_{lag}'] = df_e2['log_harga'].shift(lag)

# Rolling statistics berbasis t-1 biar konsisten dengan baseline dan menghindari leakage
df_e2['rolling_mean_7']  = df_e2['log_harga'].shift(1).rolling(7).mean()
df_e2['rolling_std_7']   = df_e2['log_harga'].shift(1).rolling(7).std()
df_e2['rolling_mean_14'] = df_e2['log_harga'].shift(1).rolling(14).mean()
df_e2['rolling_std_14']  = df_e2['log_harga'].shift(1).rolling(14).std()

# Kalender
df_e2['day_of_week'] = df_e2.index.dayofweek
df_e2['month'] = df_e2.index.month
df_e2['quarter'] = df_e2.index.quarter

# Spread makro
df_e2['spread_macro'] = df_e2['bi_7drr_rate'] - df_e2['inflasi_yoy_id']

# Lag eksogen
df_e2['log_xau_lag1'] = df_e2['log_xau'].shift(1)
df_e2['log_kurs_lag1'] = df_e2['log_kurs'].shift(1)

# Drop NaN setelah semua fitur dibuat
df_e2 = df_e2.dropna().copy()

# Safety check
missing_cols_e2 = [c for c in FEATURE_COLS if c not in df_e2.columns]
if missing_cols_e2:
    raise KeyError(f'Kolom FEATURE_COLS belum ada di df_e2: {missing_cols_e2}')

split_idx_e2 = int(len(df_e2) * TRAIN_RATIO)
train_e2 = df_e2.iloc[:split_idx_e2].copy()
test_e2  = df_e2.iloc[split_idx_e2:].copy()

print(f'Train: {len(train_e2)} | Test: {len(test_e2)}')
print(f'Jumlah fitur boosting: {len(FEATURE_COLS)}')

Train: 1304 | Test: 559
Jumlah fitur boosting: 20


In [ ]:
# SARIMAX baseline + optuna tuning
sarimax_e2 = SARIMAX(
    train_e2['log_harga'], exog=train_e2[EXOG_SARIMAX],
    order=SARIMAX_ORDER, seasonal_order=SARIMAX_SEAS,
    enforce_stationarity=False, enforce_invertibility=False, freq='D'
).fit(disp=False, maxiter=200)
train_e2['residual'] = train_e2['log_harga'].values - sarimax_e2.fittedvalues.values

X_train_e2 = train_e2[FEATURE_COLS].values
y_train_e2 = train_e2['residual'].values

def objective_xgb_e2(trial):
    params = {
        'n_estimators' : trial.suggest_int('n_estimators', 100, 500),
        'learning_rate' : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth' : trial.suggest_int('max_depth', 3, 8),
        'subsample' : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha' : trial.suggest_float('reg_alpha', 1e-4, 10, log=True),
        'reg_lambda' : trial.suggest_float('reg_lambda', 1e-4, 10, log=True),
        'random_state': seed, 'n_jobs': -1,
    }
    scores = []
    for tr_idx, va_idx in tscv.split(X_train_e2):
        m = xgb.XGBRegressor(**params)
        m.fit(X_train_e2[tr_idx], y_train_e2[tr_idx],
              eval_set=[(X_train_e2[va_idx], y_train_e2[va_idx])], verbose=False)
        scores.append(mean_squared_error(y_train_e2[va_idx], m.predict(X_train_e2[va_idx])))
    return np.mean(scores)

def objective_lgb_e2(trial):
    params = {
        'n_estimators' : trial.suggest_int('n_estimators', 100, 500),
        'learning_rate' : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth' : trial.suggest_int('max_depth', 3, 8),
        'num_leaves' : trial.suggest_int('num_leaves', 20, 100),
        'subsample' : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha' : trial.suggest_float('reg_alpha', 1e-4, 10, log=True),
        'reg_lambda' : trial.suggest_float('reg_lambda', 1e-4, 10, log=True),
        'random_state': seed, 'n_jobs': -1, 'verbose': -1,
    }
    scores = []
    for tr_idx, va_idx in tscv.split(X_train_e2):
        m = lgb.LGBMRegressor(**params)
        m.fit(X_train_e2[tr_idx], y_train_e2[tr_idx],
              eval_set=[(X_train_e2[va_idx], y_train_e2[va_idx])],
              callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(period=-1)])
        scores.append(mean_squared_error(y_train_e2[va_idx], m.predict(X_train_e2[va_idx])))
    return np.mean(scores)

study_xgb_e2 = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=seed))
study_xgb_e2.optimize(objective_xgb_e2, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
best_xgb_params_e2 = {**study_xgb_e2.best_params, 'random_state': seed, 'n_jobs': -1}

study_lgb_e2 = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=seed))
study_lgb_e2.optimize(objective_lgb_e2, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
best_lgb_params_e2 = {**study_lgb_e2.best_params, 'random_state': seed, 'n_jobs': -1, 'verbose': -1}

print(f'Best XGB (2021): {best_xgb_params_e2}')
print(f'Best LGB (2021): {best_lgb_params_e2}')

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

Best XGB (2021): {'n_estimators': 337, 'learning_rate': 0.011711509955524094, 'max_depth': 6, 'subsample': 0.6682096494749166, 'colsample_bytree': 0.6260206371941118, 'reg_alpha': 5.5517216852447255, 'reg_lambda': 6.732248920775331, 'random_state': 42, 'n_jobs': -1}
Best LGB (2021): {'n_estimators': 479, 'learning_rate': 0.010124083959432524, 'max_depth': 8, 'num_leaves': 21, 'subsample': 0.9895680834513327, 'colsample_bytree': 0.9967271775876176, 'reg_alpha': 8.55839209640483, 'reg_lambda': 6.957700515599782, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}


In [ ]:
# Direct Multi-Step Hybrid Forecast
all_results_e2  = []
metrics_rows_e2 = []

for h in range(1, MAX_HORIZON + 1):
    print(f'(Exp2-2021) Horizon H+{h}')
    train_pairs = []
    for t in range(len(train_e2) - h):
        history_y    = train_e2['log_harga'].iloc[:t+1]
        history_exog = train_e2[EXOG_SARIMAX].iloc[:t+1]
        future_exog  = train_e2[EXOG_SARIMAX].iloc[t+1:t+1+h]
        if len(future_exog) < h:
            continue
        try:
            m_hist = SARIMAX(
                history_y, exog=history_exog,
                order=SARIMAX_ORDER, seasonal_order=SARIMAX_SEAS,
                enforce_stationarity=False, enforce_invertibility=False, freq='D'
            ).fit(disp=False, maxiter=200)
            fc_h       = m_hist.forecast(steps=h, exog=future_exog).iloc[-1]
            residual_h = train_e2['log_harga'].iloc[t + h] - fc_h
            train_pairs.append((train_e2[FEATURE_COLS].iloc[t].values, residual_h))
        except Exception:
            continue

    X_tr = np.array([p[0] for p in train_pairs])
    y_tr = np.array([p[1] for p in train_pairs])

    xgb_h_e2 = xgb.XGBRegressor(**best_xgb_params_e2)
    xgb_h_e2.fit(X_tr, y_tr)
    lgb_h_e2 = lgb.LGBMRegressor(**best_lgb_params_e2)
    lgb_h_e2.fit(X_tr, y_tr)

    history_wf = train_e2.copy()
    sarimax_preds_log, actual_vals, feat_rows = [], [], []

    for i in range(len(test_e2) - h + 1):
        exog_future = test_e2[EXOG_SARIMAX].iloc[i:i+h]
        if len(exog_future) < h:
            break
        try:
            m = SARIMAX(
                history_wf['log_harga'], exog=history_wf[EXOG_SARIMAX],
                order=SARIMAX_ORDER, seasonal_order=SARIMAX_SEAS,
                enforce_stationarity=False, enforce_invertibility=False, freq='D'
            ).fit(disp=False, maxiter=200)
            fc = m.forecast(steps=h, exog=exog_future)
            sarimax_preds_log.append(fc.iloc[-1])
            feat_rows.append(test_e2[FEATURE_COLS].iloc[i].values)
            actual_vals.append(test_e2['harga_emas_antam_idr'].iloc[i + h - 1])
            history_wf = pd.concat([history_wf, test_e2.iloc[[i]]])
        except Exception:
            continue

    sarimax_preds_log = np.array(sarimax_preds_log)
    X_feat = np.array(feat_rows)
    actual_arr = np.array(actual_vals)

    pred_s_e2 = np.exp(sarimax_preds_log)
    pred_xgb_e2 = np.exp(sarimax_preds_log + xgb_h_e2.predict(X_feat))
    pred_lgb_e2 = np.exp(sarimax_preds_log + lgb_h_e2.predict(X_feat))

    res_s_e2   = evaluate_forecast(actual_arr, pred_s_e2)
    res_xgb_e2 = evaluate_forecast(actual_arr, pred_xgb_e2)
    res_lgb_e2 = evaluate_forecast(actual_arr, pred_lgb_e2)

    all_results_e2.append({
        'horizon': h, 'actual': actual_arr,
        'sarimax_preds': pred_s_e2, 'xgb_preds': pred_xgb_e2, 'lgb_preds': pred_lgb_e2,
        'sarimax': res_s_e2, 'hybrid_xgb': res_xgb_e2, 'hybrid_lgb': res_lgb_e2,
    })
    metrics_rows_e2.extend([
        {'horizon': f'H+{h}', 'model': 'SARIMAX (2021)', 'MAE': res_s_e2['mae'], 'RMSE': res_s_e2['rmse'], 'R2': res_s_e2['r2']},
        {'horizon': f'H+{h}', 'model': 'Hybrid XGB (2021)', 'MAE': res_xgb_e2['mae'], 'RMSE': res_xgb_e2['rmse'], 'R2': res_xgb_e2['r2']},
        {'horizon': f'H+{h}', 'model': 'Hybrid LGB (2021)', 'MAE': res_lgb_e2['mae'], 'RMSE': res_lgb_e2['rmse'], 'R2': res_lgb_e2['r2']},
    ])
    print(f"H+{h} | SARIMAX MAE:{res_s_e2['mae']:>10,.0f} R2:{res_s_e2['r2']:>6.3f} | XGB MAE:{res_xgb_e2['mae']:>10,.0f} R2:{res_xgb_e2['r2']:>6.3f} | LGB MAE:{res_lgb_e2['mae']:>10,.0f} R2:{res_lgb_e2['r2']:>6.3f}")

(Exp2-2021) Horizon H+1
H+1 | SARIMAX MAE:    13,351 R2: 0.996 | XGB MAE:    13,264 R2: 0.996 | LGB MAE:    13,264 R2: 0.996
(Exp2-2021) Horizon H+2
H+2 | SARIMAX MAE:    19,588 R2: 0.994 | XGB MAE:    19,321 R2: 0.994 | LGB MAE:    19,321 R2: 0.994
(Exp2-2021) Horizon H+3
H+3 | SARIMAX MAE:    24,182 R2: 0.992 | XGB MAE:    23,702 R2: 0.992 | LGB MAE:    23,702 R2: 0.992
(Exp2-2021) Horizon H+4
H+4 | SARIMAX MAE:    28,751 R2: 0.990 | XGB MAE:    27,971 R2: 0.990 | LGB MAE:    27,971 R2: 0.990
(Exp2-2021) Horizon H+5
H+5 | SARIMAX MAE:    32,359 R2: 0.987 | XGB MAE:    31,447 R2: 0.988 | LGB MAE:    31,447 R2: 0.988
(Exp2-2021) Horizon H+6
H+6 | SARIMAX MAE:    36,036 R2: 0.985 | XGB MAE:    34,668 R2: 0.986 | LGB MAE:    34,668 R2: 0.986
(Exp2-2021) Horizon H+7
H+7 | SARIMAX MAE:    39,237 R2: 0.982 | XGB MAE:    38,146 R2: 0.983 | LGB MAE:    38,146 R2: 0.983


## Export Metrics


In [ ]:
# export metrics Experiment 2
metrics_exp2 = pd.DataFrame(metrics_rows_e2)
metrics_exp2 = metrics_exp2.sort_values(['horizon', 'model']).reset_index(drop=True)
metrics_exp2.to_csv('metrics_exp2_data_2021.csv', index=False)
metrics_exp2


,horizon,model,MAE,RMSE,R2
0,H+1,Hybrid LGB (2021),13263.687044,25667.163563,0.996227
1,H+1,Hybrid XGB (2021),13263.687043,25667.163562,0.996227
2,H+1,SARIMAX (2021),13351.417123,25745.535563,0.996204
3,H+2,Hybrid LGB (2021),19320.920759,33362.558104,0.993618
4,H+2,Hybrid XGB (2021),19320.920762,33362.558106,0.993618
5,H+2,SARIMAX (2021),19587.715966,33582.649173,0.993533
6,H+3,Hybrid LGB (2021),23702.046654,36640.358197,0.992294
7,H+3,Hybrid XGB (2021),23702.046641,36640.358184,0.992294
8,H+3,SARIMAX (2021),24182.117495,37080.975469,0.992107
9,H+4,Hybrid LGB (2021),27970.884360,41848.316136,0.989938
